In [388]:
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil
import pandas as pd

In [389]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [390]:
from mediapipe import solutions
from mediapipe.framework.formats import landmark_pb2
import numpy as np

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, hand_detection_result, pose_detection_result):
  hand_landmarks_list = hand_detection_result.hand_landmarks
  handedness_list = hand_detection_result.handedness
  pose_landmarks_list = pose_detection_result.pose_landmarks
  
  annotated_image = np.copy(rgb_image)

  # Loop through the detected poses to visualize.
  for idx in range(len(pose_landmarks_list)):
    pose_landmarks = pose_landmarks_list[idx]

    # Draw the pose landmarks.
    pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
    pose_landmarks_proto.landmark.extend([
      landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in pose_landmarks
    ])
    solutions.drawing_utils.draw_landmarks(
      annotated_image,
      pose_landmarks_proto,
      solutions.pose.POSE_CONNECTIONS,
      solutions.drawing_styles.get_default_pose_landmarks_style())

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
    hand_landmarks_proto.landmark.extend([
      landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in hand_landmarks
    ])
    solutions.drawing_utils.draw_landmarks(
      annotated_image,
      hand_landmarks_proto,
      solutions.hands.HAND_CONNECTIONS,
      solutions.drawing_styles.get_default_hand_landmarks_style(),
      solutions.drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)
    
    
  return annotated_image

In [391]:
detection_confidence = 0.5
tracking_confidence = 0.5

In [392]:
# STEP 2: Create an HandLandmarker object.
base_options = mp.tasks.BaseOptions(model_asset_path='hand_landmarker.task')
options = mp.tasks.vision.HandLandmarkerOptions(base_options=base_options,
                                                num_hands=2,
                                                min_tracking_confidence=tracking_confidence,
                                                min_hand_detection_confidence=detection_confidence)
hand_detector = mp.tasks.vision.HandLandmarker.create_from_options(options)

In [393]:
# STEP 2: Create an PoseLandmarker object.
base_options = mp.tasks.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(base_options=base_options,
                                        output_segmentation_masks=False,
                                        min_pose_detection_confidence=detection_confidence,
                                        min_tracking_confidence=tracking_confidence)
pose_detector = vision.PoseLandmarker.create_from_options(options)

In [4263]:
def find_points(img, depth_img=None, *, hand_detection_result, pose_detection_result):
    depth_img = np.array(depth_img) / 255 if depth_img is not None else None
    height, width, _ = img.shape

    pose_landmarks_list = pose_detection_result.pose_landmarks    
    pose_coord = np.array([10000000] * 2 * 6).reshape((6,2))
    pose_pixel_coord = np.array([10000000] * 2 * 6).reshape((6,2))
    pose_coord[0] = pose_pixel_coord[0] = [-10000000, -10000000]
    
    if len(pose_landmarks_list) > 0:
        pose_landmarks = pose_landmarks_list[0]
        pose_ids = [16, 14, 12, 11, 13, 15]
        pose_coord = np.array([(landmark.x, landmark.y) for landmark in pose_landmarks])[pose_ids]
        pose_pixel_coord = np.minimum(np.floor(pose_coord * np.array([width, height])), np.array([width-1, height-1])).astype(int)
    

    hand_pixel_coord1, hand_pixel_coord2 = None, None
    hand_coord1, hand_coord2 = None, None
    hand_landmarks_list = hand_detection_result.hand_landmarks

    if len(hand_landmarks_list) == 1:
        hand_landmarks = hand_landmarks_list[0]
        hand_coord = np.array([(landmark.x, landmark.y) for landmark in hand_landmarks])
        hand_pixel_coord = np.minimum(np.floor(hand_coord * np.array([width, height])), np.array([width-1, height-1])).astype(int)

        if np.linalg.norm(hand_pixel_coord[0] - pose_pixel_coord[0]) < np.linalg.norm(hand_pixel_coord[0] - pose_pixel_coord[-1]):
            hand_pixel_coord1 = hand_pixel_coord
            hand_coord1 = hand_coord
        else:
            hand_pixel_coord2 = hand_pixel_coord
            hand_coord2 = hand_coord
        
    elif len(hand_landmarks_list) >= 2:
        hand_landmarks = hand_landmarks_list[1]
        hand_coord = np.array([(landmark.x, landmark.y) for landmark in hand_landmarks])
        hand_pixel_coord = np.minimum(np.floor(hand_coord * np.array([width, height])), np.array([width-1, height-1])).astype(int)
        hand_pixel_coord1 = hand_pixel_coord
        hand_coord1 = hand_coord

        hand_landmarks = hand_landmarks_list[0]
        hand_coord = np.array([(landmark.x, landmark.y) for landmark in hand_landmarks])
        hand_pixel_coord = np.minimum(np.floor(hand_coord * np.array([width, height])), np.array([width-1, height-1])).astype(int)
        hand_pixel_coord2 = hand_pixel_coord
        hand_coord2 = hand_coord

        if np.linalg.norm(hand_pixel_coord1[0] - pose_pixel_coord[0]) > np.linalg.norm(hand_pixel_coord1[0] - pose_pixel_coord[-1]):
            hand_pixel_coord1, hand_pixel_coord2 = hand_pixel_coord2, hand_pixel_coord1
            hand_coord1, hand_coord2 = hand_coord2, hand_coord1

    # find the depth values    
    if hand_pixel_coord1 is not None:
        d = depth_img[hand_pixel_coord1[:,1], hand_pixel_coord1[:,0]]
        hand_coord1 = np.concatenate([hand_coord1, d[:,None]], axis=1)
    else:
        hand_coord1 = np.array([np.nan] * 3 * 21).reshape((21,3))

    if hand_pixel_coord2 is not None:
        d = depth_img[hand_pixel_coord2[:,1], hand_pixel_coord2[:,0]]
        hand_coord2 = np.concatenate([hand_coord2, d[:,None]], axis=1)
    else:
        hand_coord2 = np.array([np.nan] * 3 * 21).reshape((21,3))

    if pose_pixel_coord[0][0] != -10000000:
        d = depth_img[pose_pixel_coord[:,1], pose_pixel_coord[:,0]]
        pose_coord = np.concatenate([pose_coord, d[:,None]], axis=1) 
    else:
        pose_coord = np.array([np.nan] * 3 * 6).reshape((6,3))  

    return hand_coord1, pose_coord, hand_coord2

In [413]:
def show_video_with_annotation(path, crop=None):
    capture = cv2.VideoCapture(path)

    while capture.isOpened():
        ret, frame = capture.read()
        if not ret:
            break

        # Converting the from BGR to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        if crop is not None:
            frame = frame[
                slice(crop['top_left'][1], crop['bottom_right'][1]), 
                slice(crop['top_left'][0], crop['bottom_right'][0]),
                :
            ]
            z = 3
            frame = cv2.resize(frame, None, fx=z, fy=z, interpolation=cv2.INTER_LANCZOS4)

        data = cv2.flip(frame, 1)
        frame = mp.Image(image_format=mp.ImageFormat.SRGB, data=data)

        hand_detection_result = hand_detector.detect(frame)
        pose_detection_result = pose_detector.detect(frame)

        annotated_image = draw_landmarks_on_image(frame.numpy_view(), hand_detection_result, pose_detection_result)
        cv2.imshow('video', cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break
        
    capture.release()
    cv2.destroyAllWindows()

In [442]:
def process_images(img_path, depth_path, crop=None):
    capture1 = cv2.VideoCapture(img_path)
    capture2 = cv2.VideoCapture(depth_path)

    hand1, pose, hand2 = {}, {}, {}

    frame_count = 0

    while capture1.isOpened() and capture2.isOpened():
        ret1, frame1 = capture1.read()
        ret2, frame2 = capture2.read()

        if not ret1 or not ret2:
            break

        # Converting the from BGR to RGB
        frame1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB)
        frame2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        if crop is not None:
            frame1 = frame1[
                slice(crop['top_left'][1], crop['bottom_right'][1]), 
                slice(crop['top_left'][0], crop['bottom_right'][0]),
                :
            ]
            frame2 = frame2[
                slice(crop['top_left'][1], crop['bottom_right'][1]), 
                slice(crop['top_left'][0], crop['bottom_right'][0]),
            ]
            z = 3
            frame1 = cv2.resize(frame1, None, fx=z, fy=z, interpolation=cv2.INTER_LANCZOS4)
            frame2 = cv2.resize(frame2, None, fx=z, fy=z, interpolation=cv2.INTER_LANCZOS4)

        data1 = cv2.flip(frame1, 1)
        data2 = cv2.flip(frame2, 1)

        frame1 = mp.Image(image_format=mp.ImageFormat.SRGB, data=data1)
        frame2 = mp.Image(image_format=mp.ImageFormat.GRAY8, data=data2)

        hand_detection_result = hand_detector.detect(frame1)
        pose_detection_result = pose_detector.detect(frame1)

        hand1_, pose_, hand2_ = find_points(frame1.numpy_view(), frame2.numpy_view(), hand_detection_result=hand_detection_result, pose_detection_result=pose_detection_result)

        hand1[frame_count] = hand1_
        pose[frame_count] = pose_
        hand2[frame_count] = hand2_

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

        frame_count += 1

    capture1.release()
    capture2.release()
    cv2.destroyAllWindows()

    return hand1, pose, hand2

In [1]:
task = "T002_Left\Playing_piano"     #done

In [ ]:
path = "Tim-Tremor\\" + task + "\kinect.avi"
depth = "Tim-Tremor\\" + task + "\kinect_map.avi"
crop = {
    'top_left': [141, 170],
    'bottom_right': [461, 426]
}
show_video_with_annotation(path, crop=crop)

In [4264]:
hand1, pose, hand2 = process_images(path, depth, crop=crop)

import csv
with open('hand1.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    for value in hand1.values():
        writer.writerow(value.reshape(-1))

with open('pose.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    for value in pose.values():
        writer.writerow(value.reshape(-1))

with open('hand2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    for value in hand2.values():
        writer.writerow(value.reshape(-1))

In [4265]:
thand1 = np.array(list(hand1.values()))
tpose = np.array(list(pose.values()))
thand2 = np.array(list(hand2.values()))

In [4266]:
def process_hand(hand):
    time_frame = 3 * 30
    slices = hand.shape[0] // time_frame

    hand = hand[:slices * time_frame].reshape(slices, time_frame, 21*3)

    for arr in hand:
        col_means = np.nanmean(arr, axis=0)
        inds = np.where(np.isnan(arr))
        arr[inds] = np.take(col_means, inds[1])

    return hand

def process_pose(pose):
    time_frame = 3 * 30
    slices = pose.shape[0] // time_frame

    pose = pose[:slices * time_frame].reshape(slices, time_frame, 6*3)

    for arr in pose:
        inds = np.where(np.isnan(arr))
        col_means = np.nanmean(arr, axis=0)
        arr[inds] = np.take(col_means, inds[1])

    return pose

thand1 = process_hand(thand1)
tpose = process_pose(tpose)
thand2 = process_hand(thand2)

In [ ]:
home_dir = "Tim-Tremor\TimTremorMultimodal"
lm_pose = "data\\" + task + "\pose\landmark"
lm_sensor_hand = "data\\" + task + "\sensor_hand\landmark"
lm_non_sensor_hand = "data\\" + task + "\\non_sensor_hand\landmark"

In [4268]:
acc_sensor_hand = "data\\" + task + "\sensor_hand\\acc"
acc_pose = "data\\" + task + "\pose\\acc"

In [2]:
label = 0

In [ ]:
# graph type
resting = 1
motion = 2
pose = 3

## processing sensor hand

In [4270]:
def foo_sensor_hand():
    sensor_hand = thand1
    val = []
    for i in range(1, len(sensor_hand)):
        filename = os.path.join(home_dir, lm_sensor_hand, f"{i}.csv")
        np.savetxt(filename, sensor_hand[i], delimiter=",")

        val.append(f"{os.path.join(lm_sensor_hand, f'{i}.csv')},1,{os.path.join(acc_sensor_hand, f'{i}.csv')},{label}")

    filename = os.path.join(home_dir, "graph_sensor_test.csv")
    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        for d in val:
            writer.writerow([d])

foo_sensor_hand()

## processing pose

In [4271]:
def foo_pose():
    val = []
    for i in range(1, len(tpose)):
        filename = os.path.join(home_dir, lm_pose, f"{i}.csv")
        np.savetxt(filename, tpose[i], delimiter=",")

        val.append(f"{os.path.join(lm_pose, f'{i}.csv')},3,{os.path.join(acc_pose, f'{i}.csv')},{label}")

    filename = os.path.join(home_dir, "graph_sensor_test.csv")
    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        for d in val:
            writer.writerow([d])

foo_pose()

## processing non sensor hand

In [4272]:
def foo_non_sensor_hand():
    val = []
    non_sensor_hand = thand2
    for i in range(1, non_sensor_hand.shape[0]):
        filename = os.path.join(home_dir, lm_non_sensor_hand, f"{i}.csv")
        np.savetxt(filename, non_sensor_hand[i], delimiter=",")

        val.append(f"{os.path.join(lm_non_sensor_hand, f'{i}.csv')},1,,{label}")

    filename = os.path.join(home_dir, "graph_test.csv")
    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        for d in val:
            writer.writerow([d])

foo_non_sensor_hand()

## ACC processing

In [3]:
import pandas as pd
import numpy as np
import os

In [ ]:
def process_acc(filename):
    df = pd.read_csv(filename, sep='\t', header=None)
    df.drop([3, 7, 8], axis=1, inplace=True)
    df = df.to_numpy()

    acc1 = df[:, 0:3]
    acc2 = df[:, 3:]

    time_frame = 3000
    slices = acc1.shape[0] // time_frame
    acc1 = acc1[:slices*time_frame]
    acc2 = acc2[:slices*time_frame]

    acc1 = [(acc1[i] + acc1[i+1] + acc1[i+2]) / 3 for i in range(0, len(acc1), 3)]
    acc2 = [(acc2[i] + acc2[i+1] + acc2[i+2]) / 3 for i in range(0, len(acc2), 3)]

    time_frame = 1000
    slices = len(acc1) // time_frame
    acc1 = np.array(acc1[:slices*time_frame]).reshape(-1, time_frame, 3)
    acc2 = np.array(acc2[:slices*time_frame]).reshape(-1, time_frame, 3)

    return acc1, acc2

filename = "Tim-Tremor\\" + task + "\kinect_accelerometer.tsv"
acc1, acc2 = process_acc(filename)

In [ ]:
home_dir = "Tim-Tremor\TimTremorMultimodal"
acc_pose = "data\\" + task + "\pose\\acc"
acc_sensor_hand = "data\\" + task + "\sensor_hand\\acc"
acc_non_sensor_hand = "data\\" + task + "\\non_sensor_hand\\acc"

val_file = os.path.join(home_dir, "sensor_train.csv")

val = []

In [6]:
import csv
def write_to_file(file_name, data):
    with open(file_name, mode='a', newline='') as file:
        writer = csv.writer(file)
        for d in data:
            writer.writerow([d])

In [7]:
def foo():
    for i in range(acc1.shape[0]):
        filename = os.path.join(home_dir, acc_sensor_hand, f"{i}.csv")
        np.savetxt(filename, acc1[i], delimiter=",")

        filename = os.path.join(home_dir, acc_pose, f"{i}.csv")
        np.savetxt(filename, acc2[i], delimiter=",")

        val.append(f",,{os.path.join(acc_sensor_hand, f'{i}.csv')},{label}")
        val.append(f",,{os.path.join(acc_pose, f'{i}.csv')},{label}")


foo()
write_to_file(val_file, val)